# Titans-style Memory Adapter trên Qwen2.5-0.5B (tiếng Việt)

**Mục tiêu:** minh họa *học tại thời điểm test* (test-time memorization) của bài báo *Titans: Learning to Memorize at Test Time*.

**Lưu ý trung thực:** đây **không phải** Titans end-to-end của paper. Backbone Qwen đã pretrain được **đóng băng**; ta gắn thêm một module `NeuralMemory` (thư viện `titans-pytorch`) vào hidden state ở giữa mạng và chỉ train phần mới. Hãy gọi nó là *"memory adapter kiểu Titans trên backbone đóng băng"*.

Cơ chế của memory: loss kết hợp key-value `‖M(k)−v‖²`, **surprise có momentum** (η) và **forgetting/weight decay** (α), memory là MLP sâu (Eq. 12–14 của paper).


| Pha | Việc gì xảy ra | Tham số thay đổi |
|---|---|---|
| 1. Pretrain memory (outer loop) | Train trên Wikipedia tiếng Việt, tài liệu dài chia thành segment liên tiếp; memory phải mang thông tin sang segment sau | `mem`, `out`, `gate` |
| 2. Test time (inner loop) | Đóng băng **mọi** tham số; dữ liệu mới đi qua từng segment, chỉ **trọng số fast-weight của memory** cập nhật | Chỉ trạng thái memory |

**Cách chạy trên Kaggle:** Accelerator = GPU (T4/P100), Internet = On, Run All. Checkpoint lưu ở `/kaggle/working/ckpt_<loại>/` (`last.pt` để resume, `best.pt`, `<loại>_vi_memory.pt` để dùng cho webapp và notebook so sánh). Nếu hết phiên: *Save Version* → thêm output làm Input ở phiên sau → đặt `INIT_CKPT` trỏ tới `last.pt`. Để notebook so sánh dùng đúng dữ liệu đánh giá, hãy giữ luôn file `data_vi.pt` trong output.

**Chế độ thử nhanh:** đặt `DEBUG = True` để chạy cả notebook bằng model ngẫu nhiên rất nhỏ (không cần GPU/Internet) nhằm kiểm tra code.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "titans-pytorch==0.6.0", "transformers", "datasets", "accelerate", "matplotlib"])

In [ ]:
import os, time, math, json, re, random, glob, html as _html
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

DEBUG = False   # True = chạy thử nhanh bằng model ngẫu nhiên nhỏ, không cần GPU/Internet
MEMORY_KIND = "titans"    # "titans" | "ttt_linear" | "ttt_mlp"   (đổi dòng này để train biến thể khác)

# ---- Backbone & dữ liệu (GIỮ NGUYÊN giữa các notebook để so sánh công bằng) ----
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SEG_LEN    = 256      # độ dài 1 segment = cửa sổ mà backbone nhìn thấy
N_SEGS     = 8        # số segment liên tiếp của 1 tài liệu khi train
TBPTT      = 2        # backward sau mỗi TBPTT segment (cắt gradient qua state memory)
N_TRAIN_DOCS = 6000
N_EVAL_DOCS  = 24
# Loại các bài trùng chủ đề vật lý khỏi dữ liệu train (tránh rò rỉ khi test bằng sách vật lý)
EXCLUDE_TITLE_REGEX = r"vật lý|cơ học|nhiệt động|điện từ|lượng tử|tương đối|newton|động lực học|năng lượng"

# ---- Memory (cùng kích thước cho cả hai: 4 head x 64 = 256 chiều, block/chunk = 16 như paper TTT) ----
MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD, MEM_DEPTH = 16, 4, 64, 2
USE_MOMENTUM = True     # chỉ dùng cho Titans; đặt False để chạy ablation (train lại) không có momentum
HOOK_LAYER   = None     # None = giữa mạng

# ---- Train ----
BATCH_DOCS, LR, WARMUP, MAX_STEPS = 2, 1e-3, 100, 3000
EVAL_EVERY, SAVE_EVERY, MAX_MINUTES = 200, 100, 170
CKPT_DIR  = os.environ.get("CKPT_DIR", f"/kaggle/working/ckpt_{MEMORY_KIND}")
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/working/data_vi.pt")
INIT_CKPT = None        # đường dẫn last.pt từ phiên trước (nếu muốn tiếp tục)
SEED = 0

if DEBUG:
    SEG_LEN, N_SEGS, TBPTT = 32, 4, 2
    MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD = 4, 2, 16
    N_TRAIN_DOCS, N_EVAL_DOCS, MAX_STEPS = 16, 4, 6
    WARMUP, EVAL_EVERY, SAVE_EVERY = 2, 3, 3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DEBUG:", DEBUG, "| memory:", MEMORY_KIND)
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s
    display = print

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen2Config, Qwen2ForCausalLM

if DEBUG:
    torch.manual_seed(1234)     # backbone ngẫu nhiên phải giống nhau giữa các notebook DEBUG
    _cfg = Qwen2Config(vocab_size=500, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                       num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096)
    lm = Qwen2ForCausalLM(_cfg)
    def encode(t): return [b % 500 for b in t.encode("utf-8")]
    def decode(ids): return "".join(chr(32 + (int(i) % 90)) for i in ids)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _dt = torch.float32  # Avoid FP16 overflow in the frozen backbone on T4
    try:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=_dt)
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=_dt)
    def encode(t): return tok(t, add_special_tokens=False)["input_ids"]
    def decode(ids): return tok.decode(ids)

lm = lm.to(device).eval()
for p in lm.parameters():
    p.requires_grad_(False)
print("tham số backbone (đóng băng), triệu:", round(sum(p.numel() for p in lm.parameters()) / 1e6, 1))

In [ ]:
from titans_pytorch import NeuralMemory, mem_state_detach

class MemLM(nn.Module):
    # Backbone đóng băng + một module memory gắn vào hidden state ở giữa mạng.
    # Cùng cách đọc ra (out, gate) cho mọi loại memory để so sánh công bằng.
    def __init__(self, lm, kind, layer=None):
        super().__init__()
        self.lm, self.kind = lm, kind
        d = lm.config.hidden_size
        if kind == "titans":
            # batch_size=MEM_CHUNK: sau mỗi chunk memory mới được cập nhật và chunk sau tính gradient theo trọng số MỚI
            # (mini-batch gradient descent của paper). Nếu để batch_size=None, thư viện luôn tính surprise theo trọng số
            # khởi tạo nên memory không bao giờ ảnh hưởng tới surprise.
            self.mem = NeuralMemory(dim=d, chunk_size=MEM_CHUNK, batch_size=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD,
                                    momentum=USE_MOMENTUM, default_step_transform_max_lr=INNER_LR,
                                    default_model_kwargs=dict(depth=MEM_DEPTH, expansion_factor=4.0))
            # titans-pytorch tạo tham số khởi tạo memory theo head bằng `repeat` (view stride 0) -> Adam không ghi được.
            for i, p in enumerate(self.mem.memory_model_parameters):
                self.mem.memory_model_parameters[i] = nn.Parameter(p.detach().clone().contiguous())
        elif kind in ("ttt_linear", "ttt_mlp"):
            self.mem = TTTMem(d, MEM_HEADS, MEM_DIM_HEAD, MEM_CHUNK, "linear" if kind == "ttt_linear" else "mlp")
        else:
            raise ValueError(kind)
        self.out = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.layer = layer if layer is not None else lm.config.num_hidden_layers // 2
        self.state, self.enabled, self.trace = None, True, False
        self.calls, self.w_ref = [], None
        lm.model.layers[self.layer].register_forward_hook(self._hook)

    def reset(self):
        self.state = None

    def detach_state(self):
        if self.state is None:
            return
        if self.kind == "titans":
            self.state = mem_state_detach(self.state)
        else:
            self.state = {k: v.detach() for k, v in self.state.items()}

    def fast_params(self):
        if self.kind == "titans":
            return sum(v.numel() for k, v in self.state.weights.items() if k.startswith("model.weights"))
        return sum(v.numel() for v in self.state.values())

    def trainable_parameters(self):
        return list(self.mem.parameters()) + list(self.out.parameters()) + [self.gate]

    def _mem_call(self, h):
        if self.kind == "titans":
            r, st, sur = self.mem(h, state=self.state, return_surprises=True)
            info = None
            if self.trace:
                flat = torch.cat([v.flatten(2) for k, v in st.updates.items() if k.startswith("model.weights")], dim=-1)
                info = dict(surprise=sur[0].detach().float().mean((0, 1)), lr=sur[1].detach().float().mean((0, 1)),
                            snaps=flat.detach().float())
            return r, st, info
        return self.mem(h, state=self.state, want_trace=self.trace)

    def _hook(self, module, args, output):
        if not self.enabled:
            return output
        h = output[0] if isinstance(output, tuple) else output
        r, st, info = self._mem_call(h.float())
        self.state = st
        if self.trace and info is not None:
            with torch.no_grad():
                snaps = info["snaps"]                                   # (B', n+1, P), snaps[:, 0] = trạng thái trước lần gọi
                if self.w_ref is None:
                    self.w_ref = snaps[:, 0].clone()
                dabs = (snaps - self.w_ref[:, None]).norm(dim=-1)
                dist = (dabs / self.w_ref.norm(dim=-1, keepdim=True)).mean(0)
                step = (snaps[:, 1:] - snaps[:, :-1]).norm(dim=-1).mean(0)
                self.calls.append(dict(surprise=info["surprise"].cpu().numpy(), theta=info["lr"].cpu().numpy(),
                                       dist=dist.cpu().numpy(), dist_abs=dabs.mean(0).cpu().numpy(), step=step.cpu().numpy()))
        h2 = h + (self.gate * self.out(r)).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2

def build_model(kind):
    m = MemLM(lm, kind, HOOK_LAYER)
    m.mem.to(device); m.out.to(device); m.gate.data = m.gate.data.to(device)
    return m

In [ ]:

probe_text = ("Trong m? h?nh n?y, b? nh? ghi l?i th?ng tin v? c?c ?o?n v?n b?n ?? ??c. "
              "N?ng l??ng ???c b?o to?n khi m?t h? c? l?p chuy?n t? tr?ng th?i n?y sang tr?ng th?i kh?c. ") * 150
ids = torch.tensor(encode(probe_text)[:SEG_LEN * N_SEGS], device=device).view(1, -1)
for INNER_LR in (1.0, 0.01):
    torch.manual_seed(SEED)
    model = build_model('titans')
    with torch.no_grad():
        for i, seg in enumerate(ids.split(SEG_LEN, dim=-1)):
            logits = model.lm(input_ids=seg, use_cache=False, logits_to_keep=2).logits
            weights = list(model.state.weights.values())
            print('inner cap', INNER_LR, 'segment', i, 'finite logits', bool(logits.isfinite().all()),
                  'max fast-weight', max(w.abs().max().item() for w in weights), flush=True)
    model.enabled = False
